In [1]:
import numpy as np
from elephant import spike_train_generation
from elephant import statistics
import quantities as pq
from scipy.stats import linregress
from scipy.interpolate import make_interp_spline
from neo.io import get_io
from neo import SpikeTrain
import matplotlib.patches as patches
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt


### Get file EPSP

In [6]:
listFile = ['GrC_Subject15_180116/180116_0006 EPSP.abf',
 'GrC_Subject06_090216/090216_0004 EPSP.abf',
 'GrC_Subject11_131115/131115_0006 EPSP.abf']

### Figure 2b

In [7]:
freqI=[]
freqF=[]
for file_path in listFile:
    l1=[]
    l2=[]
    data = get_io(file_path).read()
    for y, segment in enumerate(data[0].segments):
        seg=data[0].segments[y].analogsignals[0]
        time= seg.times
        spike_times=spike_train_generation.spike_extraction(seg)
        if y >=8 and y<=14: # condition on segment 
            train = SpikeTrain(spike_times, t_stop=time[-1], units='s')
            rateI = float(statistics.mean_firing_rate(train,t_start=time[0]+100*pq.ms,t_stop=time[0]+600*pq.ms).magnitude)
            rateF = float(statistics.mean_firing_rate(train,t_start=time[0]+1600*pq.ms,t_stop=time[0]+2100*pq.ms).magnitude)
            l1.append(rateI)
            l2.append(rateF)
    freqI.append(l1)
    freqF.append(l2)


listIFC=[]
for i in range(len(freqI)):
    l=[]
    for y in range(len(freqI[i])):
        a=(freqF[i][y]-freqI[i][y])
        b=freqI[i][y]
        
        ifc=(a/b)*100
        l.append(ifc)
    listIFC.append(l)



In [10]:
listIFC

[[-100.0, -100.0], [-100.0, -100.0], [-100.0, -100.0]]

In [9]:
listCurrent= list(np.arange(0,100,10))
colors = ['#3f709e', '#5fab99', '#f29744' ]
legend =["Adapting","Non-Adapting","Accelerating"]

for i in range(len(listIFC)):
    print(listIFC[i])
    model=make_interp_spline(listCurrent,listIFC[i])
    xs=np.linspace(8,20,100) #smoothing of the line
    ys=model(xs)

    plt.plot(xs, ys, color=colors[i], label=legend[i])
    plt.scatter(listCurrent,listIFC[i], color=colors[i])


ax = plt.gca()

ax.set_xticks([6, 8, 10, 12, 14, 16, 18, 20, 22])  
ax.tick_params(axis='x', which='major', size=5)  
ax.set_xticks([7, 9, 11, 13, 15, 17, 19, 21, 23], minor=True) 
ax.tick_params(axis='x', which='minor', size=3)  
ax.set_xlim(6, 22)  


ax.set_ylim(-120, 120)  
ax.set_yticks([-120, -100, -80, -60, -40, -20, 0, 20, 40, 60, 80, 100, 120])  

ax.tick_params(axis='y', which='major', size=5)  
ax.set_yticks([-110, -90, -70, -50, -30, -10, 10, 30, 50, 70, 90, 110], minor=True) 
ax.tick_params(axis='y', which='minor', size=3)  
ax.set_yticklabels(['-120', '', '-80', '', '-40', '', '0', '', '40', '', '80', '', '120'])
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.hlines(y=0, xmin=6, xmax=22, color= 'k', linestyle='--')    
plt.xlabel('Injected Current (pA)')
plt.ylabel('IFC (%)')


plt.legend()
plt.show()


[-100.0, -100.0]


ValueError: Shapes of x (10,) and y (2,) are incompatible